In [ ]:
prefix_pop = 'hdx'
prefix_adm = 'gadm'

prefix = f'{prefix_pop}_{prefix_adm}'

In [ ]:
import logging

# Create a global logger variable
logger = None

def setupLogger(logFilePath: str) -> None:
    """Sets up a global logger that logs to both console and file, with timestamps.

    Args:
        logFilePath: Path to the log file.
    """
    global logger
    logger = logging.getLogger('dualLogger')
    logger.setLevel(logging.INFO)

    if not logger.handlers:
        formatter = logging.Formatter(
            fmt='%(asctime)s - %(message)s',
            datefmt='%Y-%m-%d %H:%M:%S'
        )

        consoleHandler = logging.StreamHandler()
        consoleHandler.setFormatter(formatter)

        fileHandler = logging.FileHandler(logFilePath, mode='a', encoding='utf-8')
        fileHandler.setFormatter(formatter)

        logger.addHandler(consoleHandler)
        logger.addHandler(fileHandler)

def log(message: str) -> None:
    """Logs a message to both console and file via the global logger.

    Args:
        message: The message to be logged.
    """
    if logger is None:
        raise RuntimeError('Logger has not been initialized. Call setupLogger() first.')
    logger.info(message)

def closeLogger() -> None:
    """Closes and removes all handlers from the global logger."""
    if logger is None:
        return
    for handler in logger.handlers[:]:
        handler.close()
        logger.removeHandler(handler)

In [ ]:
setupLogger('merge.log')

log(f"Starting{'='*20}{prefix}{'='*20}")

In [ ]:
# Standard libraries
import numpy as np
import pandas as pd

# Optimization
from scipy.optimize import minimize

# Plotting
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Geospatial processing
import geopandas as gpd
from geopandas.tools import sjoin
from shapely.geometry import base, Point, Polygon, MultiPolygon, LineString
from shapely.ops import unary_union, nearest_points

# Geocoding
from geopy.geocoders import Nominatim

In [ ]:
import os
import wget

url = 'https://raw.githubusercontent.com/gromicho/tools/main/jg_folium.py'
filename = 'jg_folium.py'

# Ensure the file is removed before downloading
if os.path.exists(filename):
    os.remove(filename)

# Download the file (this will now always download the latest version)
wget.download(url, filename)

In [ ]:
import jg_folium as jg

In [ ]:
epsg = 'EPSG:4326' # https://en.wikipedia.org/wiki/World_Geodetic_System#WGS_84

wp_pop = gpd.read_parquet('../Population/wp_nl.parquet').to_crs(epsg)
hdx_pop = gpd.read_parquet('../Population/HDX_NL.parquet').to_crs(epsg)

cbs = gpd.read_parquet('../administrative/cbs.parquet').to_crs(epsg)
gadm = gpd.read_parquet('../administrative/gadm.parquet').to_crs(epsg)

In [ ]:
pop = wp_pop if prefix_pop == 'wp' else hdx_pop
nl = cbs if prefix_adm == 'cbs' else gadm

In [ ]:
assert len(pop[~pop.is_valid]) == 0
assert len(nl[~nl.is_valid]) == 0

In [ ]:
def findRealOverlaps(
    gdf: gpd.GeoDataFrame,
    minOverlapArea: float = 1e4,
    nameColumn: str = 'Municipality',
    verbose: bool = False
) -> gpd.GeoDataFrame:
    """
    Find significant overlaps between nearby geometries in a GeoDataFrame.

    Parameters:
    - gdf: GeoDataFrame containing named polygon geometries.
    - minOverlapArea: Minimum overlap area in square meters to retain.
    - nameColumn: Name of the column in gdf to use for municipality labels.
    - verbose: If True, prints number of overlaps found.

    Returns:
    - GeoDataFrame with columns: index_1, index_2, muni_1, muni_2, geometry, area
    """
    gdf = gdf.to_crs('EPSG:28992').copy()
    _ = gdf.sindex  # force spatial index creation

    # Spatial join to find intersecting geometries
    candidates = sjoin(gdf, gdf, how='inner', predicate='intersects')
    candidates['index_1'] = candidates.index
    candidates['index_2'] = candidates['index_right']

    # Remove self-matches and symmetric duplicates
    candidates = candidates[candidates['index_1'] != candidates['index_2']]
    candidates['pair'] = candidates.apply(
        lambda row: tuple(sorted((row['index_1'], row['index_2']))), axis=1
    )
    candidates = candidates.drop_duplicates(subset='pair')

    geoms = gdf.geometry
    names = gdf[nameColumn]

    rows = []
    for idx1, idx2 in candidates['pair']:
        geom1 = geoms.loc[idx1]
        geom2 = geoms.loc[idx2]
        inter = geom1.intersection(geom2)
        if not inter.is_empty and inter.area > minOverlapArea:
            rows.append({
                'index_1': idx1,
                'index_2': idx2,
                'muni_1': names.loc[idx1],
                'muni_2': names.loc[idx2],
                'geometry': inter,
                'area': inter.area
            })

    if verbose:
        print(f'Found {len(rows)} valid overlaps.')

    if rows:
        return gpd.GeoDataFrame(rows, crs='EPSG:28992', geometry='geometry')
    else:
        return gpd.GeoDataFrame(columns=['index_1', 'index_2', 'muni_1', 'muni_2', 'geometry', 'area'],
                                geometry='geometry', crs='EPSG:28992')


In [ ]:
def summarizeOverlapAreas(overlaps: gpd.GeoDataFrame) -> tuple[float, float]:
    """
    Compute the total area of overlaps in two ways:
    - Sum of all pairwise overlaps (may double-count shared regions)
    - Total unique area covered by all overlaps (no double-counting)

    Parameters:
    - overlaps: GeoDataFrame from findRealOverlaps, must have 'geometry' and 'area' columns.

    Returns:
    - Tuple: (sumOfPairwiseAreas, totalUniqueOverlapArea)
    """
    if overlaps.empty:
        return 0.0, 0.0

    # 1. Sum of individual intersection areas (possibly with overcounting)
    sumOfPairwiseAreas = overlaps['area'].sum()

    # 2. Union of all overlap geometries to avoid double/triple counting
    unioned = unary_union(overlaps.geometry.values)
    totalUniqueOverlapArea = unioned.area

    return sumOfPairwiseAreas, totalUniqueOverlapArea

In [ ]:
overlaps = findRealOverlaps(nl, nameColumn='Municipality', minOverlapArea=0, verbose=True)
sumPairwise, sumUnique = summarizeOverlapAreas(overlaps)

nof_overlaps = len(overlaps)

print(f'Sum of pairwise overlaps (may overcount): {sumPairwise:,.0f} m²')
print(f'Total unique overlap area (no overcounting): {sumUnique:,.0f} m²')

In [ ]:
overlaps['area_km2'] = overlaps.area / 1e6

In [ ]:
if not overlaps.empty:
    top = overlaps.loc[overlaps.area.idxmax()]
    print(f"Largest overlap: {top.area / 1e6:.2f} km² between {nl.loc[top.index_1]['Municipality']} and {nl.loc[top.index_2]['Municipality']}")
    display(overlaps[['muni_1', 'muni_2', 'area_km2']].sort_values('area_km2',ascending=False).round(5))

In [ ]:
print(overlaps['geometry'].apply(lambda g: g.geom_type).value_counts())

In [ ]:
overlaps

In [ ]:
# Separate out individual parts from MultiPolygons and GeometryCollections
overlapsClean = overlaps.explode(index_parts=False, ignore_index=True)
overlapsClean.sort_values('area_km2',ascending=False)

In [ ]:
print(overlapsClean['geometry'].apply(lambda g: g.geom_type).value_counts())

In [ ]:
if len(overlapsClean):
    overlapsClean = overlapsClean[
        overlapsClean.geometry.apply(lambda geom: isinstance(geom, (Polygon, MultiPolygon)))
    ]
    display(overlapsClean.sort_values('area_km2',ascending=False))


In [ ]:
print(overlapsClean['geometry'].apply(lambda g: g.geom_type).value_counts())

In [ ]:
def findTripleOverlaps(overlaps: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Identify areas where 3 or more municipalities overlap.

    Parameters:
    - overlaps: GeoDataFrame with pairwise overlaps (from findRealOverlaps)

    Returns:
    - GeoDataFrame with geometry of triple (or more) overlaps and list of municipalities involved
    """
    if overlaps.empty:
        return gpd.GeoDataFrame(columns=['municipalities', 'geometry'], geometry='geometry', crs=overlaps.crs)

    # Build list of all municipalities and their intersection geometry
    records = []
    for _, row in overlaps.iterrows():
        records.append({'geometry': row['geometry'], 'municipalities': set([row['muni_1'], row['muni_2']])})
    
    # Initialize with first
    tripleZones = []

    for i, rec1 in enumerate(records):
        for j in range(i + 1, len(records)):
            rec2 = records[j]
            inter = rec1['geometry'].intersection(rec2['geometry'])
            if not inter.is_empty and isinstance(inter, Polygon) and inter.area > 0:
                combinedMuni = rec1['municipalities'].union(rec2['municipalities'])
                if len(combinedMuni) >= 3:
                    tripleZones.append({'municipalities': combinedMuni, 'geometry': inter})
    
    if tripleZones:
        return gpd.GeoDataFrame(tripleZones, geometry='geometry', crs=overlaps.crs)
    else:
        return gpd.GeoDataFrame(columns=['municipalities', 'geometry'], geometry='geometry', crs=overlaps.crs)


In [ ]:
tripleOverlaps = findTripleOverlaps(overlaps)

print(f'Found {len(tripleOverlaps)} triple-or-more overlaps.')

# Optionally inspect:
tripleOverlaps[['municipalities', 'geometry']]


In [ ]:
these = set(overlaps[overlaps.muni_1.str.contains('Capelle')].muni_2.values) | set(overlaps[overlaps.muni_1.str.contains('Capelle')].muni_1.values)
these

### Finding the Point Closest to Multiple Geometries

The function `findClosestPointToGeometries` computes the point in 2D space that minimizes the **sum of distances** to a given list of geometries (such as polygons or multipolygons). This can be interpreted as a generalization of the *geometric median* problem, extended to arbitrary shapes.

#### Problem Statement

Given a list of Shapely geometries `geoms`, we seek a point $p = (x, y)$ such that:

$$
\text{minimize } f(x, y) = \sum_{i=1}^n d(p, G_i)
$$

where:
- $d(p, G_i)$ is the shortest distance from point $p$ to geometry $G_i$,
- $G_i \in \text{geoms}$ for $i = 1, \dots, n$.

#### Method

1. **Objective Function**: Takes a 2D point and returns the sum of distances to all geometries.
2. **Initial Guess**: Uses the average of the bounding boxes’ centers as the starting point.
3. **Optimization**: Uses `scipy.optimize.minimize` (L-BFGS-B) to minimize the objective function.
4. **Result**: Returns a Shapely `Point` if successful; `None` otherwise.

#### Our Use Case

- We will center our explorations on such points that are close to all geometries.

#### Notes

- Geometries should be in a **projected CRS** (e.g. EPSG:28992 or EPSG:3857) for distance accuracy.
- Invalid geometries may cause optimization to fail.
- This minimizes **sum of distances**, not the distance to centroids or containment.


In [ ]:
def pointToShapeDistance(xy: np.ndarray, shapes: list[base.BaseGeometry]) -> float:
    """Objective function: sum of distances from (x, y) to each shape."""
    p = Point(xy)
    return sum(p.distance(shape) for shape in shapes)

def findClosestPointToGeometries(geoms: list[base.BaseGeometry]) -> Point:
    """Finds the point minimizing the sum of distances to all geometries."""
    if not geoms:
        print('Empty list of geometries')
        return None

    # Check if all geometries are valid
    if not all(g.is_valid for g in geoms):
        print('[Warning] One or more geometries are invalid.')

    # Use centroid of bounding box of all shapes as initial guess
    all_bounds = np.array([geom.bounds for geom in geoms])
    center_x = all_bounds[:, [0, 2]].mean()
    center_y = all_bounds[:, [1, 3]].mean()
    x0 = np.array([center_x, center_y])

    result = minimize(pointToShapeDistance, x0, args=(geoms,), method='L-BFGS-B')
    
    if result.success:
        return Point(result.x)
    else:
        print('[Optimization failed]')
        print('Message:', result.message)
        print('Last evaluated point:', result.x)
        print('Final objective value:', result.fun)
        return None


In [ ]:
municipalities_to_show = these - {'Rotterdam'}
colors_to_use = { m : c for m, c in zip(municipalities_to_show, ['blue', 'green', 'orange'])}
colors_to_use

In [ ]:
these, municipalities_to_show

In [ ]:
nl_of_interest = nl[nl.Municipality.isin(municipalities_to_show)].copy()
nl_of_interest['color'] = nl_of_interest.Municipality.map(colors_to_use)

In [ ]:
geoms_to_use = nl_of_interest.geometry.values.tolist()
closest_point = findClosestPointToGeometries(geoms_to_use)
closest_point

In [ ]:
m = None
if closest_point:
    m = nl_of_interest.explore(location=[closest_point.y, closest_point.x],color=nl_of_interest.color,zoom_start=14)
    jg.FoliumToPng(m, 'overlaps')
m

In [ ]:
fig, ax = plt.subplots(figsize=(30, 30))
nl.to_crs(overlapsClean.crs).plot(ax=ax, color='lightgrey', alpha=.5, edgecolor='grey', linewidth=0.1)
if len(overlapsClean):
    overlapsClean.plot(ax=ax, color='yellow', edgecolor='red', alpha=0.7,linewidth=0.8)
# plt.title('Cleaned and Filtered Overlaps')
plt.axis('off')
plt.savefig('overlaps_cleaned.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
m = None
if len(overlapsClean):
    m = overlapsClean.explore()
m

In [ ]:
# Make sure both are in the same projected CRS
points = pop.to_crs('EPSG:28992')
overlaps = overlaps.to_crs('EPSG:28992')  # result from findRealOverlaps

# Use spatial join to identify points that fall within any overlap geometry
pointsInOverlap = gpd.sjoin(points, overlaps, how='inner', predicate='within')

print(f'{len(pointsInOverlap)} points lie in the overlap regions.')

# If you want the unique set of points:
uniquePoints = points.loc[pointsInOverlap.index.unique()]

uniquePoints.Population.sum()

In [ ]:
m = None
if len(uniquePoints):
    centroid = uniquePoints.to_crs(4326).union_all().centroid
    center_coords = (centroid.y, centroid.x)
    m = uniquePoints.explore(location=center_coords, zoom_start=8)
    jg.FoliumToPng(m, 'overlaps_population_over_NL')
m

In [ ]:
if len(pointsInOverlap):
    # Make sure the pair is always in the same order (to avoid duplicates like A–B and B–A)
    pointsInOverlap['muni_pair'] = pointsInOverlap.apply(
        lambda row: tuple(sorted((row['muni_1'], row['muni_2']))), axis=1
    )

    # Count how many points per unique pair
    pairCounts = pointsInOverlap['muni_pair'].value_counts().reset_index()
    pairCounts.columns = ['muni_pair', 'point_count']

    # Optional: split back into two columns
    pairCounts[['muni_1', 'muni_2']] = pd.DataFrame(pairCounts['muni_pair'].tolist(), index=pairCounts.index)
    pairCounts = pairCounts.drop(columns='muni_pair')

    # Display result
    display(pairCounts.sort_values('point_count', ascending=False))

In [ ]:
sum_pop_in_overlap_capelle_rotterdam = 0
if len(pointsInOverlap):
    sum_pop_in_overlap_capelle_rotterdam = pointsInOverlap[pointsInOverlap.muni_pair == ('Capelle aan den IJssel', 'Rotterdam')].Population.sum()
sum_pop_in_overlap_capelle_rotterdam

In [ ]:
def show_points_municipalities(
    pop, 
    adm, 
    targets,
    alpha
):
    """
    Display points within a given municipality on a map.

    Args:
        pop (gpd.GeoDataFrame): Population data with geometries.
        adm (gpd.GeoDataFrame): Administrative boundaries.
        municipality_name (str): Name of the municipality to filter.
        municipality_color (str, optional): Color for the municipality 
            boundary. Default is 'green'.
        points_color (str, optional): Color for the points. Default is 'red'.
        alpha (float, optional): Transparency level for the municipality. 
            Default is 0.25.

    Returns:
        folium.Map: Interactive map showing the municipality and the 
            points within it.
    """

    gdf = pop.to_crs(adm.crs)
    m = None

    for municipality_name, color in targets.items():
        municipality = adm[adm.Municipality == municipality_name]
        points_in_municipality = gdf[
            gdf.geometry.within(municipality.geometry.union_all())
        ]
        log(f'{points_in_municipality.shape[0]:,} points in {municipality_name} {points_in_municipality.Population.sum():,.2f}')

        m = municipality.explore(m=m, color=color['municipality'], alpha=alpha)
        m = points_in_municipality.explore(m=m, color=color['points'])
    
    return m

In [ ]:
targets = { 
    'Capelle aan den IJssel' : { 'municipality' : 'green', 'points' : 'red'},   
    'Krimpen aan den IJssel' : { 'municipality' : 'yellow', 'points' : 'blue'}
}

In [ ]:
m = show_points_municipalities( pop, nl, targets, alpha=.2 )
jg.FoliumToPng(m, 'hdx_cbs_capelle_krimpen')
m

In [ ]:
targets = { 
    'Nissewaard' : { 'municipality' : 'green', 'points' : 'red'},   
    'Rotterdam' : { 'municipality' : 'yellow', 'points' : 'blue'}
}

In [ ]:
m = show_points_municipalities( pop, nl, targets, alpha=.2 )

In [ ]:
assert pop.crs == nl.crs

In [ ]:
log( f"Performing a 'left join within' from {len(pop):,} points" )
res = gpd.sjoin(
    pop, 
    nl, 
    how='left', 
    predicate='within'
).drop(columns=['index_right'])
log( f"resulted in {len(res):,} rows from the {len(pop):,} given rows")

In [ ]:
if len(res.index) > res.index.nunique():
    log( f'{len(res.index) - res.index.nunique():,} duplicate matches found' )
    res = res[~res.index.duplicated(keep='first')]
    log( 'Keeping only the first' )
assert len(res.index) == res.index.nunique()

In [ ]:
assert len(pop) == len(res)

In [ ]:
missing = pop[res.Municipality.isna()]
log( f'Municipality missing for {len(missing):,} out of {len(pop):,} points' )
log( f'{len(missing)/len(pop)*100:.2f}% points')
log( f'{missing.Population.sum()/pop.Population.sum()*100:.2f}% headcount')

In [ ]:
ax = nl.plot(
    color=nl.color_by_city, 
    edgecolor="black", 
    figsize=(13,8), 
    alpha=.5
)  # Plot polygons
missing.plot(
    ax=ax, 
    color="red", 
    markersize=5
)  # Plot missing points
plt.axis('off')
plt.savefig(f'{prefix}_missing.png', format="png", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# use a projected CRS (e.g., EPSG:28992 for meters)
projected_crs = 'EPSG:28992'

# Perform nearest join and get the distance
nearest_match = gpd.sjoin_nearest(
    missing.to_crs(projected_crs), 
    nl.to_crs(projected_crs), 
    how="left", 
    distance_col="meters"
)

# Convert back to WGS 84 if needed
nearest_match = nearest_match.to_crs(pop.crs)

In [ ]:
nearest_match.meters.hist()

In [ ]:
furthest = nearest_match.loc[nearest_match.meters.idxmax()]

furthest_point = furthest.name  # Index of the furthest point
furthest_municipality = furthest['Municipality']  # Municipality name

f'Point {furthest_point} is {nearest_match.meters.max():,.0f} meters from the closest municipality {furthest_municipality}'

In [ ]:
lat, lon = pop.loc[furthest_point][['latitude','longitude']]

geolocator = Nominatim(user_agent="analytics notes")
location = geolocator.reverse((lat, lon), exactly_one=True, timeout=10)

log( 'The furthest point from any municipality is at:')
log( location.address )

In [ ]:
# Extract polygon and point
polygon = nl.loc[nl.Municipality == furthest_municipality, 'geometry'].values[0]
point = nearest_match.loc[nearest_match.meters.idxmax(), 'geometry']
nearest_point = nearest_points(point, polygon.boundary)[1]

# Check if nearest_point is a polygon vertex
if polygon.geom_type == "MultiPolygon":
    polygon_vertices = [tuple(pt) for poly in polygon.geoms for pt in poly.exterior.coords]
else:
    polygon_vertices = list(polygon.exterior.coords)

rounded_nearest = tuple(round(c, 6) for c in nearest_point.coords[0])
rounded_vertices = [tuple(round(c, 6) for c in v) for v in polygon_vertices]

if rounded_nearest in rounded_vertices:
    log("Nearest point is a vertex of a municipality polygon.")
else:
    log("Nearest point is on an edge of a municipality polygon.")

# Create segment and compute distance
shortest_segment = LineString([point.coords[0], nearest_point.coords[0]])

point_gdf = gpd.GeoDataFrame(geometry=[point], crs="EPSG:4326").to_crs("EPSG:28992")
nearest_point_gdf = gpd.GeoDataFrame(geometry=[nearest_point], crs="EPSG:4326").to_crs("EPSG:28992")
distance_meters = point_gdf.distance(nearest_point_gdf).values[0]

# Create circle buffer and convert to WGS84
circle = point_gdf.buffer(distance_meters)
circle_gdf = gpd.GeoDataFrame(geometry=circle, crs="EPSG:28992").to_crs("EPSG:4326")

# Convert layers to WGS84
polygon_gdf = gpd.GeoDataFrame(geometry=[polygon], crs="EPSG:4326")
segment_gdf = gpd.GeoDataFrame(geometry=[shortest_segment], crs="EPSG:4326")
segment_gdf["Shortest Distance"] = distance_meters
point_gdf = point_gdf.to_crs("EPSG:4326")

log(f'{distance_meters:,.2f} meters distance')

# Compute bounding box
all_geoms = gpd.GeoSeries(
    pd.concat([
        polygon_gdf.geometry,
        point_gdf.geometry,
        gpd.GeoSeries([nearest_point], crs="EPSG:4326"),
        segment_gdf.geometry,
        circle_gdf.geometry
    ]),
    crs="EPSG:4326"
)

union_geom = all_geoms.union_all()
minx, miny, maxx, maxy = union_geom.bounds
bounding_box = [[miny, minx], [maxy, maxx]]

# Build map
m = polygon_gdf.explore(color="blue", style_kwds={'fillOpacity': 0.1}, bbox=bounding_box)
m = point_gdf.explore(m=m, color="red", marker_kwds={'radius': 5})
m = segment_gdf.explore(m=m, color="green", tooltip="Shortest Distance")
m = circle_gdf.explore(m=m, color="purple", style_kwds={'fillOpacity': 0.1, 'fillColor': 'purple'})

m.fit_bounds(bounding_box)

jg.FoliumToPng(m, f'{prefix}_furthest')
m


In [ ]:
nof_match = nearest_match.index.value_counts().to_dict()
multiple = [k for k,v in nof_match.items() if v > 1]

In [ ]:
m = None
if multiple:
    dubious = nearest_match.loc[multiple[0]]
    m = nl[nl.Municipality.isin(dubious.Municipality)].explore()
    m = dubious.explore(m=m, color="red", marker_kwds={'radius': 3})
    jg.FoliumToPng( m, f'{prefix}_multiple' )
m  # this will display the map if it was created

In [ ]:
# Keep only the nearest match for each index
nearest_match = nearest_match.sort_values('meters').groupby(nearest_match.index).first()

In [ ]:
res.update(nearest_match)

In [ ]:
res.groupby('Municipality')['color_by_city'].nunique().max()

In [ ]:
df = res.loc[missing.index].copy()

In [ ]:
ax = nl.plot(
    color=nl.color_by_city, 
    edgecolor="black", 
    figsize=(13,8), 
    alpha=.5
)  # Plot polygons
df.plot(
    ax=ax, 
    color=df.color_by_city, 
    markersize=8
)  # Plot missing points
plt.axis('off')
plt.savefig(f'{prefix}_matched.png', format="png", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
def optimizeCategoricalColumns(gdf: gpd.GeoDataFrame, threshold: float = 0.5) -> gpd.GeoDataFrame:
    """
    Converts string columns to categorical if their unique-to-total ratio is below a threshold.

    Args:
        gdf: Input GeoDataFrame.
        threshold: Max ratio of unique values / total rows to convert to category.

    Returns:
        Optimized GeoDataFrame with selected columns converted to 'category'.
    """
    gdf_opt = gdf.copy()
    for col in gdf_opt.select_dtypes(include='object').columns:
        nunique_ratio = gdf_opt[col].nunique() / len(gdf_opt)
        if nunique_ratio < threshold:
            gdf_opt[col] = gdf_opt[col].astype('category')
    return gdf_opt


In [ ]:
original_memory = sum(res[col].memory_usage(deep=True) for col in res.columns)
log(f'Original memory usage: {original_memory:,} bytes')

In [ ]:
res_cat = optimizeCategoricalColumns(res, threshold=0.5)

In [ ]:
categorized_memory = sum(res_cat[col].memory_usage(deep=True) for col in res_cat.columns)
log(f'Memory usage after categorizing: {categorized_memory:,} bytes')

In [ ]:
res.shape, res_cat.shape

In [ ]:
res[[col for col in res.columns if 'geo'in col]]

In [ ]:
res.memory_usage(deep=True).sort_values(ascending=False).to_frame('bytes').style.format('{:,.0f}').background_gradient(cmap='viridis', low=1, high=0)

In [ ]:
res_cat.memory_usage(deep=True).sort_values(ascending=False).to_frame('bytes').style.format('{:,.0f}').background_gradient(cmap='viridis', low=1, high=0)

In [ ]:
res_cat_simple = res_cat[list(res_cat.columns[:10])+['Municipality','Province']]

In [ ]:
(
    res.memory_usage(deep=True).sum() / 1e6,  # in MB
    res_cat.memory_usage(deep=True).sum() / 1e6,  # in MB
    res_cat_simple.memory_usage(deep=True).sum() / 1e6  # in MB
)


In [ ]:
res_cat_simple.to_pickle(f'{prefix}.pkl')

In [ ]:
res_cat_simple.to_feather(f'{prefix}.feather')

In [ ]:
res_cat_simple.to_parquet(f'{prefix}.parquet', engine='pyarrow')

In [ ]:
test = gpd.read_parquet(f'{prefix}.parquet')

In [ ]:
test.equals(res_cat_simple)

In [ ]:
test.memory_usage(deep=True).sort_values(ascending=False).to_frame('bytes').style.format('{:,.0f}').background_gradient(cmap='Greens', low=1, high=0)

In [ ]:
for col in test.select_dtypes(include='float'):
    test[col] = pd.to_numeric(test[col], downcast='float')

for col in test.select_dtypes(include='integer'):
    test[col] = pd.to_numeric(test[col], downcast='integer')

test.reset_index(drop=True, inplace=True)

In [ ]:
df = test.memory_usage(deep=True).sort_values(ascending=False).to_frame('bytes')
df['MB'] = df['bytes'] / (1024 ** 2)
df.to_latex('mem_usage.tex')
df.style.format('{:,.0f}').background_gradient(cmap='Oranges', low=1, high=0)

In [ ]:
def valuesEqual(df1: pd.DataFrame, df2: pd.DataFrame) -> bool:
    if df1.shape != df2.shape or list(df1.columns) != list(df2.columns):
        return False
    for col in df1.columns:
        a = df1[col]
        b = df2[col]
        if pd.api.types.is_numeric_dtype(a):
            if not np.allclose(a, b, equal_nan=True):
                return False
        else:
            if not a.equals(b):
                return False
    return True


In [ ]:
(test.index.values == res_cat_simple.index.values).all()

In [ ]:
test.to_pickle(f'_{prefix}.pkl')

In [ ]:
valuesEqual(test.reset_index(drop=True), res_cat_simple.reset_index(drop=True))


In [ ]:
(
    test.memory_usage(deep=True).sum() / 1e6,  # in MB
    res_cat_simple.memory_usage(deep=True).sum() / 1e6  # in MB
)

In [ ]:
def exportMemoryUsagePDF(df: pd.DataFrame, filename='mem_usage_table.pdf', cmap='Oranges'):
    mem = df.memory_usage(deep=True).sort_values(ascending=False).to_frame('bytes')
    mem['MB'] = (mem['bytes'] / 1024**2).astype(int)

    norm = mcolors.Normalize(vmin=mem['MB'].min(), vmax=mem['MB'].max())
    colormap = plt.get_cmap(cmap)

    fig, ax = plt.subplots(figsize=(8, 0.45 * len(mem) + 1))
    ax.axis('off')

    cellText = [[f'{row.bytes:,}', f'{row.MB}'] for _, row in mem.iterrows()]
    rowLabels = mem.index.to_list()
    colLabels = ['bytes', 'MB']

    table = ax.table(cellText=cellText,
                     rowLabels=rowLabels,
                     colLabels=colLabels,
                     cellLoc='right',
                     loc='center')

    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.15, 1.15)

    for i, mb in enumerate(mem['MB']):
        table[(i + 1, 1)].set_facecolor(colormap(norm(mb)))

    with PdfPages(filename) as pdf:
        pdf.savefig(fig, bbox_inches='tight')


In [ ]:
exportMemoryUsagePDF(test, filename='mem_usage_test.pdf', cmap='Oranges')

In [ ]:
log(f"Finishing{'='*20}{prefix}{'='*20}")

closeLogger()